# Complete Colab hyperparameter search

Primary user-facing notebook for selecting hyperparameters. It keeps search-space validation, calibration, estimation, execution, resumption, Pareto analysis, benchmark comparison, and export modular by calling `hpo` package APIs.

## 1. Setup

In [1]:
from pathlib import Path
import os, sys, subprocess, shutil, json
REPO_URL="https://github.com/TrueRottweiler/WashingtonCsed504.git"; BRANCH="feature/hpo-framework"; REPO_ROOT=Path("/content/WashingtonCsed504")
if not (REPO_ROOT/"src/a1-cv/hpo").exists():
    if REPO_ROOT.exists(): shutil.rmtree(REPO_ROOT)
    subprocess.run(["git","clone","--branch",BRANCH,"--single-branch",REPO_URL,str(REPO_ROOT)],check=True)
CV_DIR=REPO_ROOT/"src/a1-cv"; os.chdir(CV_DIR); sys.path.insert(0,str(CV_DIR)) if str(CV_DIR) not in sys.path else None
subprocess.run([sys.executable,"-m","pip","install","-q","-r","hpo_requirements.txt"],check=True)
import torch, optuna, pandas as pd
print(sys.version); print(torch.__version__, torch.version.cuda); print(optuna.__version__)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
2.11.0+cu128 12.8
4.9.0


## 2. Persistence and Google Drive

In [2]:
MOUNT_DRIVE=True
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST_ROOT=Path("/content/drive/MyDrive/WashingtonCsed504-HPO")
else:
    PERSIST_ROOT=Path("/content/WashingtonCsed504-HPO")
PERSIST_ROOT.mkdir(parents=True,exist_ok=True)
RESUME=True
print("Persistence root:",PERSIST_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Persistence root: /content/drive/MyDrive/WashingtonCsed504-HPO


## 3. Hardware profile, recommendations, and overrides

In [3]:
from hpo.hardware import detect_hardware
from hpo.scheduler import plan_resources
hardware=detect_hardware(); print(json.dumps(hardware.to_dict(),indent=2,default=str))
DEVICE_OVERRIDE="auto"; CONCURRENT_TRIALS_OVERRIDE=None; INTRAOP_THREADS_OVERRIDE=None; INTEROP_THREADS_OVERRIDE=None; WORKERS_OVERRIDE=None; MEMORY_RESERVE_GB=1.0
resource_plan=plan_resources(hardware,device=DEVICE_OVERRIDE,requested_concurrency=CONCURRENT_TRIALS_OVERRIDE,requested_intraop_threads=INTRAOP_THREADS_OVERRIDE,requested_interop_threads=INTEROP_THREADS_OVERRIDE,requested_workers=WORKERS_OVERRIDE,memory_reserve_gb=MEMORY_RESERVE_GB)
print(json.dumps(resource_plan.to_dict(),indent=2))

{
  "operating_system": "Linux 6.6.122+",
  "notebook": true,
  "colab": true,
  "python_version": "3.12.13",
  "torch_version": "2.11.0+cu128",
  "cuda_available": true,
  "cuda_version": "12.8",
  "cudnn_version": 91900,
  "gpu_count": 1,
  "gpus": [
    {
      "index": 0,
      "name": "Tesla T4",
      "total_vram_gb": 14.56317138671875,
      "available_vram_gb": 14.5634765625,
      "compute_capability": [
        7,
        5
      ]
    }
  ],
  "fp16_supported": true,
  "bf16_supported": true,
  "bf16_native": false,
  "tf32_supported": false,
  "mps_available": false,
  "physical_cpu_cores": 1,
  "logical_cpu_threads": 2,
  "available_ram_gb": 11.180465698242188,
  "total_ram_gb": 12.671417236328125,
  "storage_free_gb": 65.58876037597656,
  "torch_compile_available": true,
  "multiprocessing_start_method": null,
  "nvidia_smi": {
    "gpus": [
      {
        "index": 0,
        "name": "Tesla T4",
        "memory_total_mb": 15360.0,
        "memory_free_mb": 14913.0,
     

## 4. Experiment selection

In [4]:
DATASET="cifar10"  # cifar10, cifar100, imagenet32
MODEL="resnet18"   # resnet18, resnet50, vit, vit_base
if DATASET=="imagenet32":
    IMAGENET32_ROOT="/content/imagenet32"  # must already contain the repository-supported data
else: IMAGENET32_ROOT=None
NUM_CLASSES={"cifar10":10,"cifar100":100,"imagenet32":1000}[DATASET]
print(DATASET,MODEL,NUM_CLASSES)

cifar10 resnet18 10


## 5. Search-space source: built-in, Python dictionary, Python list, CSV upload, JSON/YAML, or manual input

In [5]:
from pathlib import Path

from hpo.search_space import (
    normalize_space,
    load_csv,
    load_space,
    preview_rows,
    combination_count,
)
from hpo.notebook_api import (
    preview_dataframe,
    optional_widgets,
)


INPUT_SOURCE = "builtin"
# Options:
# builtin, dictionary, list, csv, csv_upload,
# json, yaml, manual


BUILTIN_CSV = CV_DIR / (
    "hpo_configs/search_spaces/vit_cifar.csv"
    if MODEL.startswith("vit")
    else
    "hpo_configs/search_spaces/resnet18_cifar.csv"
)


DICT_SPACE = {
    "learning_rate": {
        "type": "float",
        "low": 1e-4,
        "high": 0.2,
        "log": True,
        "default": 0.01,
    },
    "batch_size": {
        "type": "categorical",
        "choices": [64, 128, 256],
        "default": 128,
    },
    "optimizer": {
        "type": "categorical",
        "choices": ["sgd", "adamw"],
        "default": "sgd",
    },
    "momentum": {
        "type": "float",
        "low": 0.8,
        "high": 0.95,
        "step": 0.05,
        "default": 0.9,
        "condition": 'optimizer == "sgd"',
    },
    "beta1": {
        "type": "float",
        "low": 0.8,
        "high": 0.95,
        "step": 0.05,
        "default": 0.9,
        "condition": 'optimizer == "adamw"',
    },
}


LIST_SPACE = [
    {
        "name": name,
        **value,
    }
    for name, value in DICT_SPACE.items()
]


MANUAL_SPACE = DICT_SPACE.copy()


if INPUT_SOURCE in {"builtin", "csv"}:
    specs = load_csv(BUILTIN_CSV)

elif INPUT_SOURCE == "csv_upload":
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "CSV upload is available in Colab. "
            "Outside Colab, use INPUT_SOURCE='csv'."
        ) from exc

    uploaded = files.upload()

    if len(uploaded) != 1:
        raise ValueError(
            "Upload exactly one CSV search-space file."
        )

    from hpo.notebook_api import normalize_uploaded_csv

    filename, content = next(iter(uploaded.items()))

    specs = normalize_uploaded_csv(
        content,
        filename=filename,
    )

elif INPUT_SOURCE == "dictionary":
    specs = normalize_space(
        DICT_SPACE,
        source_name="dictionary cell",
    )

elif INPUT_SOURCE == "list":
    specs = normalize_space(
        LIST_SPACE,
        source_name="list cell",
    )

elif INPUT_SOURCE == "manual":
    specs = normalize_space(
        MANUAL_SPACE,
        source_name="manual cell",
    )

elif INPUT_SOURCE == "json":
    specs = load_space(
        Path("/content/search_space.json")
    )

elif INPUT_SOURCE == "yaml":
    specs = load_space(
        Path("/content/search_space.yaml")
    )

else:
    raise ValueError(
        f"Unsupported INPUT_SOURCE: {INPUT_SOURCE!r}"
    )


parameter_names = [
    spec.name
    for spec in specs
]

assert parameter_names, (
    "The normalized search space is empty."
)

assert "inline" not in parameter_names, (
    "Malformed inline search-space wrapper detected."
)


display(
    preview_dataframe(specs)
)

finite_combinations = combination_count(specs)

if finite_combinations is None:
    print(
        "Finite combinations: not finite or not "
        "meaningfully enumerable because the space "
        "contains continuous parameters."
    )
else:
    print(
        "Finite combinations:",
        finite_combinations,
    )

widgets = optional_widgets()

if widgets is None:
    display("ipywidgets unavailable")
else:
    display(widgets)

print(
    "Normalized parameters:",
    parameter_names,
)
print(
    "Search-space source:",
    INPUT_SOURCE,
)

,name,type,low,high,choices,step,log,default,condition,enabled,source,item,description
0,optimizer,categorical,NaN,NaN,"[sgd, adamw]",NaN,False,sgd,None,True,/content/WashingtonCsed504/src/a1-cv/hpo_confi...,2,Optimizer family
1,learning_rate,float,0.000010,0.3000,[],NaN,True,0.1,None,True,/content/WashingtonCsed504/src/a1-cv/hpo_confi...,3,Base learning rate
2,batch_size,categorical,NaN,NaN,"[64, 128, 256, 512]",NaN,False,256,None,True,/content/WashingtonCsed504/src/a1-cv/hpo_confi...,4,Global batch size
3,weight_decay,float,0.000001,0.1000,[],NaN,True,0.0005,None,True,/content/WashingtonCsed504/src/a1-cv/hpo_confi...,5,Weight decay
4,momentum,float,0.800000,0.9900,[],0.0100,False,0.9,optimizer == 'sgd',True,/content/WashingtonCsed504/src/a1-cv/hpo_confi...,6,SGD momentum
5,nesterov,bool,NaN,NaN,"[False, True]",NaN,False,True,optimizer == 'sgd',True,/content/WashingtonCsed504/src/a1-cv/hpo_confi...,7,Nesterov acceleration
6,dampening,fixed,NaN,NaN,[],NaN,False,0.0,optimizer == 'sgd',True,/content/WashingtonCsed504/src/a1-cv/hpo_confi...,8,SGD dampening
7,beta1,float,0.800000,0.9900,[],0.0100,False,0.9,"optimizer in ['adam', 'adamw']",True,/content/WashingtonCsed504/src/a1-cv/hpo_confi...,9,Adam beta1
8,beta2,float,0.950000,0.9999,[],0.0001,False,0.999,"optimizer in ['adam', 'adamw']",True,/content/WashingtonCsed504/src/a1-cv/hpo_confi...,10,Adam beta2
9,epsilon,categorical,NaN,NaN,"[1e-08, 1e-07]",NaN,False,0.0,"optimizer in ['adam', 'adamw']",True,/content/WashingtonCsed504/src/a1-cv/hpo_confi...,11,Adam epsilon


Finite combinations: not finite or not meaningfully enumerable because the space contains continuous parameters.


Normalized parameters: ['optimizer', 'learning_rate', 'batch_size', 'weight_decay', 'momentum', 'nesterov', 'dampening', 'beta1', 'beta2', 'epsilon', 'scheduler', 'warmup_epochs', 'label_smoothing', 'gradient_clip', 'strong_augmentation', 'channels_last']
Search-space source: builtin


In [6]:

import hpo.search_space as search_space_module

print(
    "Loaded module:",
    search_space_module.__file__,
)

print(
    "load_csv exists:",
    hasattr(search_space_module, "load_csv"),
)

print(
    "load_space exists:",
    hasattr(search_space_module, "load_space"),
)

print(
    "load_json exists:",
    hasattr(search_space_module, "load_json"),
)

print(
    "load_yaml exists:",
    hasattr(search_space_module, "load_yaml"),
)

Loaded module: /content/WashingtonCsed504/src/a1-cv/hpo/search_space.py
load_csv exists: True
load_space exists: True
load_json exists: False
load_yaml exists: False


## 6. Validation and feasibility preview

In [7]:
from hpo.constraints import validate_candidate
from hpo.exceptions import InvalidTrialError
warnings=[]
for spec in specs:
    if spec.name=="batch_size": warnings.append("Batch values will be filtered by calibration before launch.")
print("Conditions:",[s.condition for s in specs if s.condition])
print("Defaults:",{s.name:s.default for s in specs if s.default is not None})
print("Warnings:",warnings)
# Structural model constraints are checked before allocation: ViT divisibility/patch size, optimizer-specific fields, precision, and memory calibration.

Conditions: ["optimizer == 'sgd'", "optimizer == 'sgd'", "optimizer == 'sgd'", "optimizer in ['adam', 'adamw']", "optimizer in ['adam', 'adamw']", "optimizer in ['adam', 'adamw']", "scheduler == 'cosine'"]
Defaults: {'optimizer': 'sgd', 'learning_rate': 0.1, 'batch_size': 256, 'weight_decay': 0.0005, 'momentum': 0.9, 'nesterov': True, 'dampening': 0.0, 'beta1': 0.9, 'beta2': 0.999, 'epsilon': 1e-08, 'scheduler': 'cosine', 'warmup_epochs': 5, 'label_smoothing': 0.1, 'gradient_clip': 0.0, 'strong_augmentation': False, 'channels_last': True}
Warnings: ['Batch values will be filtered by calibration before launch.']


## 7. Search mode and continuous execution

In [8]:
MODE="successive_halving"  # proxy, successive_halving, full
CONTINUOUS=True
CONTINUOUS_SETTINGS={"enabled":CONTINUOUS,"strategy":MODE,"maximum_trials":None,"maximum_wall_time_hours":8.0,"maximum_gpu_hours":None,"maximum_cpu_hours":None,"maximum_cost_usd":None,"target_validation_metric":None,"stop_after_no_improvement_trials":25,"minimum_improvement":0.0005,"pareto_stagnation_trials":25,"checkpoint_after_each_trial":True}
print(MODE,json.dumps(CONTINUOUS_SETTINGS,indent=2))

successive_halving {
  "enabled": true,
  "strategy": "successive_halving",
  "maximum_trials": null,
  "maximum_wall_time_hours": 8.0,
  "maximum_gpu_hours": null,
  "maximum_cpu_hours": null,
  "maximum_cost_usd": null,
  "target_validation_metric": null,
  "stop_after_no_improvement_trials": 25,
  "minimum_improvement": 0.0005,
  "pareto_stagnation_trials": 25,
  "checkpoint_after_each_trial": true
}


## 8. Objectives, hard constraints, and cost rates

In [9]:
OBJECTIVES=[{"name":"validation_top1","direction":"maximize","primary":True},{"name":"wall_seconds","direction":"minimize","primary":False},{"name":"peak_gpu_memory_mb","direction":"minimize","primary":False}]
CONSTRAINTS=[]  # e.g. {"name":"peak_gpu_memory_mb","operator":"<=","value":12000}
COST_RATES={"gpu_usd_per_hour":None,"cpu_usd_per_hour":None,"storage_usd_per_gb_month":None,"electricity_usd_per_kwh":None,"colab_subscription_usd":None,"colab_compute_unit_usd":None}
print(json.dumps({"objectives":OBJECTIVES,"constraints":CONSTRAINTS,"rates":COST_RATES},indent=2))

{
  "objectives": [
    {
      "name": "validation_top1",
      "direction": "maximize",
      "primary": true
    },
    {
      "name": "wall_seconds",
      "direction": "minimize",
      "primary": false
    },
    {
      "name": "peak_gpu_memory_mb",
      "direction": "minimize",
      "primary": false
    }
  ],
  "constraints": [],
  "rates": {
    "gpu_usd_per_hour": null,
    "cpu_usd_per_hour": null,
    "storage_usd_per_gb_month": null,
    "electricity_usd_per_kwh": null,
    "colab_subscription_usd": null,
    "colab_compute_unit_usd": null
  }
}


## 9. Calibration: batch size, real training steps, validation, checkpoint, and memory

In [10]:
from hpo.adapters import RepoModules, build_trial_model, build_trial_dataset
from hpo.calibration import calibrate_batch_sizes
modules=RepoModules(REPO_ROOT); device=torch.device(resource_plan.device)
dataset_cfg={"name":DATASET,"validation_fraction":0.1,"split_seed":42}; dataset_cfg.update({"root":IMAGENET32_ROOT} if IMAGENET32_ROOT else {})
bundle=build_trial_dataset(modules,dataset_cfg,{"seed":42},device)
model_cfg={"name":MODEL}
def make_batch(bs):
    x,y=next(bundle.train.epoch(bs,train=True)); return x,y
calibration=calibrate_batch_sizes(lambda: build_trial_model(modules,model_cfg,bundle.num_classes,device),make_batch,device=device,candidates=[32,64,128,256,512],precision="bf16" if hardware.bf16_native else "fp16" if device.type=="cuda" else "fp32",warmup_steps=2,measure_steps=3,channels_last=MODEL.startswith("resnet"))
print(json.dumps(calibration.to_dict(),indent=2))

{
  "measurements": [
    {
      "batch_size": 32,
      "status": "completed",
      "seconds_per_step": 0.015177232666701457,
      "examples_per_second": 2108.4212585214796,
      "peak_allocated_mb": 497.67919921875,
      "peak_reserved_mb": 530.0,
      "error": null
    },
    {
      "batch_size": 64,
      "status": "completed",
      "seconds_per_step": 0.019006146666773322,
      "examples_per_second": 3367.3316912725527,
      "peak_allocated_mb": 566.61669921875,
      "peak_reserved_mb": 586.0,
      "error": null
    },
    {
      "batch_size": 128,
      "status": "completed",
      "seconds_per_step": 0.03050145500007299,
      "examples_per_second": 4196.521116769469,
      "peak_allocated_mb": 705.9921875,
      "peak_reserved_mb": 738.0,
      "error": null
    },
    {
      "batch_size": 256,
      "status": "completed",
      "seconds_per_step": 0.057728165999985016,
      "examples_per_second": 4434.577048577404,
      "peak_allocated_mb": 996.1181640625,
    

Actually filter batch sizes using calibration

In [11]:
from dataclasses import replace


batch_spec = next(
    spec
    for spec in specs
    if spec.name == "batch_size"
)

completed_batches = {
    measurement.batch_size
    for measurement in calibration.measurements
    if measurement.status == "completed"
}

recommended_batches = [
    batch_size
    for batch_size in batch_spec.choices
    if (
        batch_size
        in calibration.recommended_candidates
        and batch_size in completed_batches
    )
]

fitting_search_batches = [
    batch_size
    for batch_size in batch_spec.choices
    if batch_size in completed_batches
]

safe_batch_choices = (
    recommended_batches
    or fitting_search_batches
)

if not safe_batch_choices:
    raise RuntimeError(
        "None of the configured search batch sizes "
        "completed calibration."
    )

preferred_default = (
    calibration.highest_throughput_batch
    if calibration.highest_throughput_batch
    in safe_batch_choices
    else safe_batch_choices[-1]
)

specs = [
    replace(
        spec,
        choices=tuple(safe_batch_choices),
        default=preferred_default,
    )
    if spec.name == "batch_size"
    else spec
    for spec in specs
]

print(
    "Original batch choices:",
    batch_spec.choices,
)
print(
    "Completed calibration batches:",
    sorted(completed_batches),
)
print(
    "Recommended search batches:",
    safe_batch_choices,
)
print(
    "New batch default:",
    preferred_default,
)

Original batch choices: (64, 128, 256, 512)
Completed calibration batches: [32, 64, 128, 256, 512]
Recommended search batches: [128, 256, 512]
New batch default: 512


## 10. Build configuration and pre-search estimate

In [13]:
import yaml, copy
from hpo.config import load_study_config
from hpo.estimation import (
    estimate_search,
    estimate_continuous_capacity,
)

RUN_TAG = "accuracy-tpe-s42-r01"
SEARCH_SEED = 42
SPLIT_SEED = 42

STUDY_NAME = (
    f"{MODEL}-{DATASET}-{MODE}-{RUN_TAG}"
)

CONFIG_PATH=PERSIST_ROOT/f"{STUDY_NAME}.yaml"
base=yaml.safe_load((CV_DIR/("hpo_configs/colab/vit_cifar10.yaml" if MODEL.startswith("vit") else "hpo_configs/colab/resnet18_cifar10_successive_halving.yaml")).read_text())

base["study"]["seed"] = SEARCH_SEED
base["dataset"]["split_seed"] = SPLIT_SEED

base["study"].update({"name":STUDY_NAME,"output_dir":str(PERSIST_ROOT),"storage_path":str(PERSIST_ROOT/f"{STUDY_NAME}.db"),"resume":RESUME}); base.setdefault("search", {})["mode"] = MODE; base["dataset"].update(dataset_cfg); base["model"]={"name":MODEL}; base["objectives"]=OBJECTIVES; base["constraints"]=CONSTRAINTS; base["cost_rates"]=COST_RATES; base["continuous"]=CONTINUOUS_SETTINGS
base["runtime"].update({"device":resource_plan.device,"concurrent_trials":resource_plan.concurrent_trials,"intraop_threads":resource_plan.intraop_threads,"interop_threads":resource_plan.interop_threads,"workers":resource_plan.workers_per_trial,"memory_reserve_gb":MEMORY_RESERVE_GB})
base["search_space"] = {
    s.name: {
        k: v
        for k, v in s.to_dict().items()
        if k not in {"name", "source", "item"}
        and v not in (None, [], ())
    }
    for s in specs
}
CONFIG_PATH.write_text(yaml.safe_dump(base,sort_keys=False)); config=load_study_config(CONFIG_PATH)

assert config.mode == MODE, (
    f"Expected mode {MODE!r}, "
    f"but configuration loaded {config.mode!r}"
)

assert all(
    spec.name != "inline"
    for spec in config.search_space
), "Malformed inline search-space wrapper remains"

print("Validated mode:", config.mode)
print(
    "Normalized parameters:",
    [spec.name for spec in config.search_space],
)

if calibration.highest_throughput_batch is None:
    failures = [
        measurement.__dict__
        for measurement in calibration.measurements
    ]

    raise RuntimeError(
        "No calibration batch completed successfully. "
        f"Measurements: {failures}"
    )

measurement=next(m for m in calibration.measurements if m.batch_size==calibration.highest_throughput_batch)
calibration_record={"batch_size":measurement.batch_size,"seconds_per_example":measurement.seconds_per_step/measurement.batch_size,"evaluation_seconds_per_epoch":0.0,"checkpoint_seconds_per_epoch":0.0,"checkpoint_size_mb":0.0,"peak_memory_mb":measurement.peak_allocated_mb or 0.0,"cpu_seconds":0.0,"elapsed_seconds":measurement.seconds_per_step}

estimate_arguments = {
    "train_examples": bundle.train_examples,
    "validation_examples": (
        bundle.validation_examples
    ),
    "calibration_records": [
        calibration_record
    ],
    "representative_batch_size": (
        measurement.batch_size
    ),
}

if config.continuous.enabled:
    estimate_payload = (
        estimate_continuous_capacity(
            config,
            **estimate_arguments,
        )
    )
else:
    estimate_payload = estimate_search(
        config,
        **estimate_arguments,
    ).to_dict()

ESTIMATE_PATH = (
    PERSIST_ROOT
    / f"{STUDY_NAME}-pre_search_estimate.json"
)

ESTIMATE_PATH.write_text(
    json.dumps(
        estimate_payload,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        estimate_payload,
        indent=2,
    )
)

Validated mode: successive_halving
Normalized parameters: ['optimizer', 'learning_rate', 'batch_size', 'weight_decay', 'momentum', 'nesterov', 'dampening', 'beta1', 'beta2', 'epsilon', 'scheduler', 'warmup_epochs', 'label_smoothing', 'gradient_clip', 'strong_augmentation', 'channels_last']
{
  "mode": "successive_halving",
  "expected_trials": 8,
  "expected_pruned_trials": 3,
  "expected_promoted_trials": 3,
  "expected_full_evaluations": 1,
  "expected_epochs": 72.0,
  "expected_steps": 5274,
  "expected_examples": 2700000.0,
  "duration_seconds": {
    "optimistic": 446.5110946291428,
    "expected": 595.3481261721904,
    "conservative": 952.5570018755047,
    "unit": "seconds",
    "method": "calibrated projection"
  },
  "gpu_hours": {
    "optimistic": 0.12403085961920632,
    "expected": 0.1653744794922751,
    "conservative": 0.26459916718764015,
    "unit": "GPU-hours",
    "method": "calibrated projection"
  },
  "cpu_hours": {
    "optimistic": 0.012403085961920635,
    "ex

## 11. Explicit launch gate and Search execution with live monitoring

Review the estimate above. The expensive search starts only after changing `START_SEARCH` to `True`.

In [14]:
START_SEARCH=False
if START_SEARCH:
    from hpo.monitoring import run_command_with_monitor
    study_dir=PERSIST_ROOT/STUDY_NAME
    log_path=PERSIST_ROOT/f"{STUDY_NAME}.log"
    command=[
        sys.executable,"-m","hpo.cli","--repo-root",str(REPO_ROOT),
        "search","--config",str(CONFIG_PATH),"--mode",MODE,
    ]
    if CONTINUOUS:
        command.append("--continuous")
    expected_records = None if CONTINUOUS else (
        config.full.trials * (len(config.full.budget.seeds)+1)
        if MODE=="full"
        else config.proxy.trials
    )
    return_code=run_command_with_monitor(
        command,cwd=CV_DIR,study_dir=study_dir,log_path=log_path,
        interval_seconds=15,timeout_seconds=None,expected_records=expected_records,
    )
    print("Return code:",return_code)
    if return_code!=0:
        print("Log tail:")
        print("\n".join(log_path.read_text(encoding="utf-8").splitlines()[-120:]))
        raise RuntimeError("Search failed; inspect the log above")
else:
    print("Search not started. Set START_SEARCH=True after reviewing the estimate.")

Streaming output truncated to the last 5000 lines.
}
{
  "elapsed_seconds": 2132.2571610010004,
  "records": 46,
  "status_counts": {
    "completed": 44,
    "seed_completed": 2
  },
  "stage_counts": {
    "proxy": 24,
    "halving": 18,
    "full": 4
  },
  "completed_candidates": 24,
  "best_validation_top1": 0.9254000186920166,
  "cumulative_gpu_hours": 0.5617765368866676,
  "cumulative_cpu_hours": 0.4731982685091667,
  "cumulative_examples": 7560000,
  "cumulative_cost_usd": 0.0,
  "last_event": "candidate_pruned",
  "projected_remaining_seconds": null
}
{
  "elapsed_seconds": 2147.293710935,
  "records": 46,
  "status_counts": {
    "completed": 44,
    "seed_completed": 2
  },
  "stage_counts": {
    "proxy": 24,
    "halving": 18,
    "full": 4
  },
  "completed_candidates": 24,
  "best_validation_top1": 0.9254000186920166,
  "cumulative_gpu_hours": 0.5617765368866676,
  "cumulative_cpu_hours": 0.4731982685091667,
  "cumulative_examples": 7560000,
  "cumulative_cost_usd": 0.0,

## 12. Resume after reconnection

In [15]:
# Reconnect, remount Drive if used, rerun setup/config cells, and execute this cell.
RESUME_SEARCH=True
if RESUME_SEARCH:
    from hpo.study import HpoStudy
    resumed=HpoStudy(CONFIG_PATH,repo_root=REPO_ROOT).run()
    print(json.dumps(resumed,indent=2,default=str))
else: print("Resume disabled. Completed trials are stored in SQLite, JSONL, state JSON, and checkpoints.")

[I 2026-07-21 09:50:00,939] Using an existing study with name 'resnet18-cifar10-successive_halving-accuracy-tpe-s42-r01' instead of creating a new one.


{
  "study": "resnet18-cifar10-successive_halving-accuracy-tpe-s42-r01",
  "mode": "continuous",
  "study_dir": "/content/drive/MyDrive/WashingtonCsed504-HPO/resnet18-cifar10-successive_halving-accuracy-tpe-s42-r01",
  "storage_path": "/content/drive/MyDrive/WashingtonCsed504-HPO/resnet18-cifar10-successive_halving-accuracy-tpe-s42-r01.db",
  "records": 192,
  "status_counts": {
    "completed": 180,
    "seed_completed": 12
  },
  "candidate_status_counts": {
    "completed": 12,
    "pruned": 84
  },
  "report": {
    "records": 192,
    "completed_candidates": 96,
    "pareto_candidates": 37,
    "plots": {
      "accuracy_vs_time": "/content/drive/MyDrive/WashingtonCsed504-HPO/resnet18-cifar10-successive_halving-accuracy-tpe-s42-r01/accuracy_vs_time.png",
      "accuracy_vs_memory": "/content/drive/MyDrive/WashingtonCsed504-HPO/resnet18-cifar10-successive_halving-accuracy-tpe-s42-r01/accuracy_vs_memory.png",
      "optimization_history": "/content/drive/MyDrive/WashingtonCsed504-HP

## 13. Results analysis and Pareto selection

In [16]:
from hpo.persistence import read_jsonl
from hpo.reporting import export_reports
from hpo.selection import highest_accuracy_under_budget, fastest_above_accuracy, lowest_memory_above_accuracy, lowest_cost_within_accuracy_margin, pareto_knee
from hpo.schemas import ObjectiveSpec
study_dir=PERSIST_ROOT/STUDY_NAME
if (study_dir/"trials.jsonl").exists():
    rows=read_jsonl(study_dir/"trials.jsonl"); objectives=[ObjectiveSpec(**o) for o in OBJECTIVES]; report=export_reports(study_dir,objectives)
    completed=[r for r in rows if r.get("status")=="completed"]
    print(report); display(pd.read_csv(study_dir/"all_trials.csv")); display(pd.read_csv(study_dir/"pareto_trials.csv"))
    print("Fastest acceptable:",fastest_above_accuracy(completed,minimum_accuracy=0.80))
    print("Lowest memory acceptable:",lowest_memory_above_accuracy(completed,minimum_accuracy=0.80))
    print("Lowest cost near best:",lowest_cost_within_accuracy_margin(completed,accuracy_margin=0.01))
    print("Pareto knee:",pareto_knee(completed,objectives))
else: print("Run or resume a study first.")

{'records': 192, 'completed_candidates': 96, 'pareto_candidates': 37, 'plots': {'accuracy_vs_time': '/content/drive/MyDrive/WashingtonCsed504-HPO/resnet18-cifar10-successive_halving-accuracy-tpe-s42-r01/accuracy_vs_time.png', 'accuracy_vs_memory': '/content/drive/MyDrive/WashingtonCsed504-HPO/resnet18-cifar10-successive_halving-accuracy-tpe-s42-r01/accuracy_vs_memory.png', 'optimization_history': '/content/drive/MyDrive/WashingtonCsed504-HPO/resnet18-cifar10-successive_halving-accuracy-tpe-s42-r01/optimization_history.png', 'proxy_vs_full': '/content/drive/MyDrive/WashingtonCsed504-HPO/resnet18-cifar10-successive_halving-accuracy-tpe-s42-r01/proxy_vs_full.png'}, 'status_counts': {'completed': 180, 'seed_completed': 12}}


,candidate_id,failure_reason,invalid_reason,metrics_accounting_scope,metrics_approximate_training_flops,metrics_approximate_training_flops_max,metrics_approximate_training_flops_min,metrics_best_validation_top1,metrics_best_validation_top1_max,metrics_best_validation_top1_min,...,resource_data_fraction,resource_evaluate_test,resource_max_steps,resource_seed,resource_stage,resource_target_epochs,resource_validation_fraction,stage,status,trial_number
0,trial-00000,NaN,NaN,NaN,7.490244e+13,NaN,NaN,0.2960,NaN,NaN,...,0.25,False,300.0,42,proxy,2,0.5,proxy,completed,0
1,trial-00001,NaN,NaN,NaN,7.490244e+13,NaN,NaN,0.1100,NaN,NaN,...,0.25,False,300.0,42,proxy,2,0.5,proxy,completed,1
2,trial-00002,NaN,NaN,NaN,7.490244e+13,NaN,NaN,0.2348,NaN,NaN,...,0.25,False,300.0,42,proxy,2,0.5,proxy,completed,2
3,trial-00003,NaN,NaN,NaN,7.490244e+13,NaN,NaN,0.1088,NaN,NaN,...,0.25,False,300.0,42,proxy,2,0.5,proxy,completed,3
4,trial-00004,NaN,NaN,NaN,7.490244e+13,NaN,NaN,0.0976,NaN,NaN,...,0.25,False,300.0,42,proxy,2,0.5,proxy,completed,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,trial-00095,NaN,NaN,NaN,4.494146e+14,NaN,NaN,0.5106,NaN,NaN,...,1.00,False,NaN,42,halving,5,1.0,halving,completed,95
188,trial-00092,NaN,NaN,NaN,1.048634e+15,NaN,NaN,0.8504,NaN,NaN,...,1.00,False,NaN,42,halving,12,1.0,halving,completed,92
189,trial-00093,NaN,NaN,NaN,1.048634e+15,NaN,NaN,0.8428,NaN,NaN,...,1.00,False,NaN,42,halving,12,1.0,halving,completed,93
190,trial-00092-full-seed42,NaN,NaN,NaN,4.494146e+15,NaN,NaN,0.9066,NaN,NaN,...,1.00,False,NaN,42,full,30,1.0,full,seed_completed,92


,candidate_id,failure_reason,invalid_reason,metrics_accounting_scope,metrics_approximate_training_flops,metrics_approximate_training_flops_max,metrics_approximate_training_flops_min,metrics_best_validation_top1,metrics_best_validation_top1_max,metrics_best_validation_top1_min,...,resource_data_fraction,resource_evaluate_test,resource_max_steps,resource_seed,resource_stage,resource_target_epochs,resource_validation_fraction,stage,status,trial_number
0,trial-00003,NaN,NaN,NaN,7.490244e+13,NaN,NaN,0.1088,NaN,NaN,...,0.25,False,300.0,42,proxy,2,0.5,proxy,completed,3
1,trial-00005,NaN,NaN,NaN,1.048634e+15,NaN,NaN,0.8540,NaN,NaN,...,1.00,False,NaN,42,halving,12,1.0,halving,completed,5
2,trial-00006,NaN,NaN,NaN,4.494146e+14,NaN,NaN,0.6528,NaN,NaN,...,1.00,False,NaN,42,halving,5,1.0,halving,completed,6
3,trial-00011,NaN,NaN,all_seed_runs,4.494146e+15,4.494146e+15,4.494146e+15,0.9256,0.9256,0.9256,...,1.00,False,NaN,42,full,30,1.0,full,completed,11
4,trial-00012,NaN,NaN,NaN,7.490244e+13,NaN,NaN,0.2052,NaN,NaN,...,0.25,False,300.0,42,proxy,2,0.5,proxy,completed,12
5,trial-00013,NaN,NaN,NaN,7.490244e+13,NaN,NaN,0.0996,NaN,NaN,...,0.25,False,300.0,42,proxy,2,0.5,proxy,completed,13
6,trial-00014,NaN,NaN,NaN,7.490244e+13,NaN,NaN,0.1100,NaN,NaN,...,0.25,False,300.0,42,proxy,2,0.5,proxy,completed,14
7,trial-00015,NaN,NaN,NaN,1.048634e+15,NaN,NaN,0.8138,NaN,NaN,...,1.00,False,NaN,42,halving,12,1.0,halving,completed,15
8,trial-00017,NaN,NaN,NaN,4.494146e+14,NaN,NaN,0.3410,NaN,NaN,...,1.00,False,NaN,42,halving,5,1.0,halving,completed,17
9,trial-00020,NaN,NaN,all_seed_runs,4.494146e+15,4.494146e+15,4.494146e+15,0.9334,0.9334,0.9334,...,1.00,False,NaN,42,full,30,1.0,full,completed,20


Fastest acceptable: {'candidate_id': 'trial-00067', 'trial_number': 67, 'stage': 'halving', 'status': 'completed', 'params': {'optimizer': 'adamw', 'learning_rate': 0.0032034283595641705, 'batch_size': 512, 'weight_decay': 2.7499987455708963e-06, 'beta1': 0.8700000000000001, 'beta2': 0.9516, 'epsilon': 1e-07, 'scheduler': 'step', 'label_smoothing': 0.15000000000000002, 'gradient_clip': 0.0, 'strong_augmentation': True, 'channels_last': True}, 'metrics': {'validation_top1': 0.853600025177002, 'validation_top5': 0.9929999709129333, 'validation_loss': 1.00583815574646, 'best_validation_top1': 0.853600025177002, 'test_top1': None, 'wall_seconds': 84.94582712299962, 'cpu_seconds': 71.25185611500001, 'gpu_hours': 0.023596063089722116, 'cpu_hours': 0.01979218225416667, 'epochs_completed': 12, 'optimization_steps': 616, 'training_examples': 315000, 'validation_examples': 35000, 'total_examples': 350000, 'parameter_count': 11173962, 'train_flops_per_image': 3328997376.0, 'approximate_training_f

## 14. Benchmark comparison

In [17]:
from hpo.baselines import load_repository_baselines
from hpo.benchmark import compare_with_reference, proxy_reliability
baselines=[b for b in load_repository_baselines(REPO_ROOT) if b.dataset==DATASET and b.model==MODEL]
print([b.__dict__ for b in baselines])
if baselines and 'completed' in locals() and completed:
    best=max(completed,key=lambda r:r.get("metrics",{}).get("validation_top1",0)); print(json.dumps(compare_with_reference(best,max(baselines,key=lambda b:b.validation_top1 or 0),accuracy_margin=0.01),indent=2,default=str))
print("Important: repository stored baselines evaluated against the final test split during training; HPO uses a deterministic validation split and reserves test for confirmation.")

[{'name': 'cifar10_resnet18', 'dataset': 'cifar10', 'model': 'resnet18', 'params': {'optimizer': 'sgd', 'batch_size': 512, 'learning_rate': 0.2, 'weight_decay': 0.0005, 'momentum': 0.9, 'nesterov': True, 'scheduler': 'cosine', 'warmup_epochs': 5, 'label_smoothing': 0.1, 'gradient_clip': 0.0, 'augmentation': 'basic', 'epochs': 30}, 'validation_top1': 0.927299976348877, 'seconds': 71.37598633766174, 'source': 'src/a1-cv/runs/cifar10_resnet18_result.json', 'measured': True}]
{
  "reference_name": "cifar10_resnet18",
  "reference_source": "src/a1-cv/runs/cifar10_resnet18_result.json",
  "reference_measured": true,
  "reference_validation_top1": 0.927299976348877,
  "discovered_validation_top1": 0.9333999752998352,
  "accuracy_gap": 0.006099998950958252,
  "within_configured_margin": true,
  "parameter_distance": {
    "components": {
      "batch_size": 1.0,
      "gradient_clip": 1.0,
      "label_smoothing": 1.0,
      "learning_rate": 1.0,
      "optimizer": 1.0,
      "scheduler": 0.0,

## 15. Export selected hyperparameters and study artifacts

In [18]:
EXPORT=True
if EXPORT and (PERSIST_ROOT/STUDY_NAME).exists():
    export_dir=PERSIST_ROOT/f"{STUDY_NAME}-export"; export_dir.mkdir(parents=True,exist_ok=True)
    for name in ["all_trials.csv","pareto_trials.csv","best_validation_configuration.json","pareto_knee_configuration.json","report_summary.json","environment.json","resolved_config.json"]:
        src=PERSIST_ROOT/STUDY_NAME/name
        if src.exists(): shutil.copy2(src,export_dir/name)
    shutil.copy2(CONFIG_PATH,export_dir/CONFIG_PATH.name); shutil.copy2(ESTIMATE_PATH,export_dir/ESTIMATE_PATH.name)
    archive=shutil.make_archive(str(export_dir),"zip",root_dir=export_dir); print("Export:",archive)
else: print("Nothing to export yet.")

Export: /content/drive/MyDrive/WashingtonCsed504-HPO/resnet18-cifar10-successive_halving-accuracy-tpe-s42-r01-export.zip


## Notes and limitations

- Search estimates are projections, not guarantees.
- Colab GPU assignment and session duration vary at runtime.
- One active trial per GPU is the default; multi-trial one-GPU execution requires measured benefit.
- Multi-objective staged promotion uses the designated primary metric; Pareto selection is applied to completed survivors.
- “Best” means best observed for the declared split, search space, fidelity, objectives, constraints, seeds, and budget.